In [10]:
import cv2

video_path = 'video_4.mp4'

frame_number_to_save = 4


cap = cv2.VideoCapture(video_path)


cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number_to_save)

ret, frame = cap.read()

output_filename = f'frame_{frame_number_to_save}.jpg'
cv2.imwrite(output_filename, frame)

cap.release()

In [27]:
import cv2
import numpy as np

video_path = "video_1.mp4"
mask_path = "mask_lights_1.png"


mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
_, mask_bin = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)
contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if len(contours) == 0:
    print("Маска не содержит контуров.")
    exit()

x, y, w, h = cv2.boundingRect(contours[0])


cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Видео закончилось или ошибка при чтении")
        break

    roi_cropped = frame[y:y+h, x:x+w]
    hsv = cv2.cvtColor(roi_cropped, cv2.COLOR_BGR2HSV)


    red_mask1 = cv2.inRange(hsv, (0, 50, 50), (10, 255, 255))
    red_mask2 = cv2.inRange(hsv, (160, 50, 50), (180, 255, 255))
    red_mask = cv2.bitwise_or(red_mask1, red_mask2)

    green_mask = cv2.inRange(hsv, (40, 50, 50), (85, 255, 255))


    red_score = cv2.countNonZero(red_mask)
    green_score = cv2.countNonZero(green_mask)

    threshold = 30  

    if red_score > green_score and red_score > threshold:
        signal_color = "red"
    elif green_score > threshold:
        signal_color = "green"
    else:
        signal_color = "none"


    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(frame, f"Signal: {signal_color}", (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("Frame", frame)
    cv2.imshow("ROI", roi_cropped)
    
    # cv2.imshow("Red mask", red_mask)
    # cv2.imshow("Green mask", green_mask)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Видео закончилось или ошибка при чтении
